# Homework 9 — Deep Neural-Network Regression

**Coverage:** Lectures 24–25<br>
**Due:** Sunday, November 22, 2026, 11:59 p.m. ET<br>
**Total:** 100 points

## Instructions

- Complete this notebook in Google Colab.
- Problem 1 is a manual mathematics problem. Show every important
  step in Markdown/LaTeX, or insert one clearly legible image of
  your handwritten derivation. Code may check arithmetic only
  after the derivation is complete.
- Problem 2 is a scaffolded scientific-computing study. Use the
  supplied random seeds and do not delete setup, helper, or check
  cells.
- Your submitted notebook must run from beginning to end in a
  fresh Colab runtime without Google Drive, absolute paths, or
  additional package installation.
- Label plots and include documented units. If a legacy dataset
  has no documented units, label the quantity as normalized or
  unit-unspecified rather than inventing units. Unless stated
  otherwise, report numerical answers to at least four
  significant digits.

## Student details

- **First name:**
- **Last name:**
- **Purdue email:**


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import copy
import random
import torch
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

SEED = 53909
rng = np.random.default_rng(SEED)
sns.set_theme(style="ticks", context="notebook")
plt.rcParams["figure.dpi"] = 120
np.set_printoptions(precision=6, suppress=True)

# During drafting this points to master. Before release, the instructor
# will replace DATA_REVISION with the immutable course release tag.
DATA_REVISION = "master"
DATA_BASE = (
    "https://raw.githubusercontent.com/PredictiveScienceLab/"
    f"data-analytics-se/{DATA_REVISION}/lecturebook/data/homework"
)

def course_data(name):
    local_candidates = [
        Path("../data/homework") / name,
        Path("lecturebook/data/homework") / name,
    ]
    for local in local_candidates:
        if local.exists():
            return local
    return f"{DATA_BASE}/{name}"


## Problem 1 — One neural-network update (25 points)

For \(h=\tanh(w_hx+b_h)\), \(\widehat y=w_oh+b_o\), and
\(L=\tfrac12(\widehat y-y)^2\), use
\(x=.5,y=1.2,w_h=.8,b_h=-.1,w_o=1.5,b_o=.2\).

1. Calculate preactivation, activation, prediction, and loss. **(7)**
2. Count trainable scalar parameters. **(3)**
3. Derive/evaluate \(\partial L/\partial\widehat y\). **(4)**
4. Given gradient `(-.386438,-.772876,-.164018,-.563031)` in
   parameter order \((w_h,b_h,w_o,b_o)\), take one SGD step with
   learning rate 0.05. **(7)**
5. Recompute prediction/loss and decide whether it is a descent
   step. **(4)**


> **Response:** Replace this text with your work.


## Problem 2 — Airfoil self-noise regression (75 points)

The UCI Airfoil Self-Noise dataset (DOI 10.24432/C5VW2C, CC BY
4.0) has 1,503 observations, five inputs, and sound-pressure level
in dB. Use PyTorch already present in Colab; install no framework.


In [ ]:
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
airfoil = pd.read_csv(course_data("hw11_airfoil_self_noise.csv"))
TARGET = "sound_pressure_db"
FEATURES = [c for c in airfoil.columns if c != TARGET]
X = airfoil[FEATURES].to_numpy(np.float32)
y = airfoil[TARGET].to_numpy(np.float32)
idx = np.arange(len(airfoil))
train_idx, remainder_idx = train_test_split(
    idx, test_size=0.30, random_state=SEED, shuffle=True
)
valid_idx, test_idx = train_test_split(
    remainder_idx, test_size=0.50, random_state=SEED + 1, shuffle=True
)
assert (len(train_idx), len(valid_idx), len(test_idx)) == (1052, 225, 226)
airfoil.head()


In [ ]:
def make_model():
    return torch.nn.Sequential(
        torch.nn.Linear(5, 16),
        torch.nn.Tanh(),
        torch.nn.Linear(16, 16),
        torch.nn.Tanh(),
        torch.nn.Linear(16, 1),
    )

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def train_network(model, X_train_scaled, y_train_scaled,
                  X_valid_scaled, y_valid_scaled,
                  max_epochs=250, patience=25,
                  min_delta=1e-6, seed=SEED):
    '''Seeded mini-batch trainer with validation early stopping.'''
    Xtr = torch.as_tensor(X_train_scaled, dtype=torch.float32)
    ytr = torch.as_tensor(y_train_scaled, dtype=torch.float32).reshape(-1, 1)
    Xva = torch.as_tensor(X_valid_scaled, dtype=torch.float32)
    yva = torch.as_tensor(y_valid_scaled, dtype=torch.float32).reshape(-1, 1)
    dataset = torch.utils.data.TensorDataset(Xtr, ytr)
    generator = torch.Generator().manual_seed(seed)
    loader = torch.utils.data.DataLoader(
        dataset, batch_size=64, shuffle=True, generator=generator
    )
    optimizer = torch.optim.Adam(
        model.parameters(), lr=1e-3, weight_decay=1e-5
    )
    loss_fn = torch.nn.MSELoss()
    best_loss = np.inf
    best_state = copy.deepcopy(model.state_dict())
    epochs_without_improvement = 0
    history = {"train": [], "valid": []}

    for _ in range(max_epochs):
        model.train()
        batch_losses = []
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            batch_losses.append(float(loss.detach()))
        model.eval()
        with torch.no_grad():
            valid_loss = float(loss_fn(model(Xva), yva))
        history["train"].append(float(np.mean(batch_losses)))
        history["valid"].append(valid_loss)
        if valid_loss < best_loss - min_delta:
            best_loss = valid_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_state)
    return model, history, best_loss

assert count_parameters(make_model()) == 385


### 2.1 Inspect, split, and transform (10 points)

Describe all variables and units. Verify disjoint split indices.
Fit separate standard scalers to training predictors and training
target only; transform validation/test without refitting.


In [ ]:
# YOUR CODE HERE


### 2.2 Linear baseline (15 points)

Fit `LinearRegression` on standardized training data. Report
validation RMSE/MAE on the original dB scale and plot residuals.
Keep the test set untouched.


In [ ]:
# YOUR CODE HERE


### 2.3 Neural model and training (20 points)

Inspect the supplied `5 → 16 → 16 → 1` tanh network and verify 385
trainable parameters. Use the supplied seeded trainer (Adam,
learning rate `1e-3`, weight decay `1e-5`, batch size 64, MSE,
250 epochs, patience 25, `min_delta=1e-6`). Confirm that it restores
the best validation checkpoint. Plot training and validation loss
and justify the stopping epoch.


In [ ]:
# YOUR CODE HERE


### 2.4 One final test evaluation (15 points)

Freeze every decision, then evaluate both models on test data once.
Report RMSE/MAE in dB, plot observed versus predicted and residuals
versus prediction, and state whether the neural model materially
improves the baseline.


In [ ]:
# YOUR CODE HERE


### 2.5 Sparse-support trust check (15 points)

In standardized input space, compute each test point's distance to
its nearest training point. Compare neural-network test RMSE in the
upper distance quartile with all remaining points. Decide whether
sparse-support predictions are trustworthy and name one limitation
hidden by aggregate RMSE.


In [ ]:
# YOUR CODE HERE


> **Response:** Give your scientific conclusion in 150 words or fewer.
